# BBO Function 1 — Part 2 reflection analysis

This notebook is for the **first 2D unknown function**.

It is designed to help answer the Part 2 reflection prompts:
- which evaluated inputs behaved like support vectors or boundary points;
- how surrogate gradients change with the inputs;
- how classification framing separates “good” and “bad” outputs;
- whether linear regression, SVM, or neural networks are most useful;
- which variables influence the surrogate most;
- whether the neural network captures nonlinear patterns better than simpler models.

Assumption: this BBO objective is being **minimised**.


## Version 4: reflection report generator

Run this after filling in the latest output. It prints draft wording for Part 2.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load original function 1 data.
# Run this notebook from the same folder where initial_data/function_1 exists.
input_data = np.load('../../data/initial_data/function_1/initial_inputs.npy')
output_data = np.load('../../data/initial_data/function_1/initial_outputs.npy')

# Add your latest evaluated point here.
# You said the new input is (0.5, 0.5). Replace NEW_OUTPUT with the portal output.
NEW_POINT = np.array([
    [0.759048, 0.965375],
    [0.714910, 0.858164],
    [0.5, 0.5],
])

NEW_OUTPUT = np.array([
    0.29645067004988124,
    0.4598605438093154,
    2.6752879910742468e-9
])

if NEW_OUTPUT is not None:
    input_data = np.vstack([input_data, NEW_POINT])
    output_data = np.append(output_data, NEW_OUTPUT)

df = pd.DataFrame(input_data, columns=['x1', 'x2'])
df['y'] = output_data
df['log_abs_y'] = np.log(np.abs(output_data) + 1e-300)
df['rank_min'] = df['y'].rank(method='min', ascending=True).astype(int)

print(df.sort_values('y').to_string(index=True))
print("\nBest point so far:")
print(df.loc[df['y'].idxmin()])


          x1        x2              y   log_abs_y  rank_min
4   0.650114  0.681526  -3.606063e-03   -5.625139         1
5   0.410437  0.147554  -2.159249e-54 -123.569835         2
6   0.312691  0.078723  -2.089093e-91 -208.798513         3
3   0.840353  0.264732  3.341771e-124 -284.314051         4
8   0.082507  0.403488   3.606771e-81 -185.226580         5
0   0.319404  0.762959   1.322677e-79 -181.624565         6
9   0.883890  0.582254   6.229856e-48 -108.694731         7
1   0.574329  0.879898   1.033078e-46 -105.886371         8
7   0.683418  0.861057   2.535001e-40  -91.173210         9
2   0.731024  0.733000   7.710875e-16  -34.798730        10
10  0.500000  0.500000   2.675288e-09  -19.739209        11

Best point so far:
x1           0.650114
x2           0.681526
y           -0.003606
log_abs_y   -5.625139
rank_min     1.000000
Name: 4, dtype: float64


In [3]:

# This cell computes simple evidence summaries that can be inserted into your reflection.

best_idx = int(np.argmin(output_data))
best_point = input_data[best_idx]
best_y = output_data[best_idx]

threshold = np.quantile(output_data, 0.25)
distance_to_threshold = np.abs(output_data - threshold)
support_indices = np.argsort(distance_to_threshold)[:min(3, len(output_data))]

print("Best observed point:")
print(f"index {best_idx}: x = [{best_point[0]:.6f}, {best_point[1]:.6f}], y = {best_y:.6e}")

print("\nSupport-vector-like / boundary-like points:")
for idx in support_indices:
    print(f"index {idx}: x = [{input_data[idx,0]:.6f}, {input_data[idx,1]:.6f}], y = {output_data[idx]:.6e}, distance to good/bad threshold = {distance_to_threshold[idx]:.6e}")

print("\nGood/bad threshold used for classification:")
print(f"good if y <= {threshold:.6e}")


Best observed point:
index 4: x = [0.650114, 0.681526], y = -3.606063e-03

Support-vector-like / boundary-like points:
index 3: x = [0.840353, 0.264732], y = 3.341771e-124, distance to good/bad threshold = 1.044547e-91
index 6: x = [0.312691, 0.078723], y = -2.089093e-91, distance to good/bad threshold = 1.044547e-91
index 8: x = [0.082507, 0.403488], y = 3.606771e-81, distance to good/bad threshold = 3.606771e-81

Good/bad threshold used for classification:
good if y <= -1.044547e-91


In [4]:

reflection = f"""
For function 1, I treated the objective as a minimisation problem and updated the data set with the latest evaluated point, including the point (0.5, 0.5) once its output was available. The best observed input so far was index {best_idx}, x = [{best_point[0]:.6f}, {best_point[1]:.6f}], with output {best_y:.6e}. This gave me a concrete exploitation region, but I did not want to query only around the current best point because the data set is still small.

To identify support-vector-like inputs, I reframed the observations as a classification problem: “good” points were the lowest-output observations and “bad” points were the rest. The most support-vector-like points were the observations closest to this good/bad threshold, because they lie near the empirical decision boundary. In this run, the closest boundary-like indices were {list(map(int, support_indices))}. Recognising these points is useful because they show where the model is most uncertain about whether a region is promising. A next query near this boundary can be valuable if I want to refine the boundary, while a query near the current best region is more exploitative.

I used surrogate models to understand the shape of the function rather than only ranking the observed outputs. A Gaussian-process surrogate was useful because it provided both a predicted mean and uncertainty, allowing me to use expected improvement for minimisation. This balances searching near low predicted outputs with exploring uncertain regions.

I also considered the BBO task as a classification problem. Logistic regression gives an interpretable linear boundary, so it is useful as a baseline, but it may be too simple if the good region is curved or local. An SVM with an RBF kernel can capture a nonlinear boundary and is closer to the “support vector” idea, but with few observations it can overfit. A neural network is the most flexible and can model nonlinear patterns, but it is harder to tune and interpret reliably with such a small data set.

For the neural-network surrogate, backpropagation gives gradients of the predicted output with respect to the inputs. These gradients indicate which input direction the model thinks would reduce the output fastest. If one coordinate has consistently larger absolute gradients, I would treat that variable as more influential and prioritise experiments that vary it more carefully. However, because the surrogate is trained on very few function evaluations, I would use the gradients as qualitative guidance rather than as proof of the true function shape.

Overall, I found the GP and SVM-style views most useful for guiding the next query. The GP is best for query selection because expected improvement explicitly balances exploitation and exploration. The SVM/classification view is best for explaining where the boundary between good and bad outputs appears to lie. The neural network is useful for discussing nonlinear approximation and gradients, but its extra flexibility is only worthwhile if I am careful not to overinterpret it.
"""
print(reflection)



For function 1, I treated the objective as a minimisation problem and updated the data set with the latest evaluated point, including the point (0.5, 0.5) once its output was available. The best observed input so far was index 4, x = [0.650114, 0.681526], with output -3.606063e-03. This gave me a concrete exploitation region, but I did not want to query only around the current best point because the data set is still small.

To identify support-vector-like inputs, I reframed the observations as a classification problem: “good” points were the lowest-output observations and “bad” points were the rest. The most support-vector-like points were the observations closest to this good/bad threshold, because they lie near the empirical decision boundary. In this run, the closest boundary-like indices were [3, 6, 8]. Recognising these points is useful because they show where the model is most uncertain about whether a region is promising. A next query near this boundary can be valuable if I wa